# IFCB PID List and Streaming Summary builder

This notebook leverages existing scripts to build a pid list and then scrape the dashboard for summary statistics about these pids 


Edit the dataset, date range, times, and output path in the setup cell before running.
Can also be used with prebuilt and imported pid lists

In [13]:
from pathlib import Path
import json
import sys

import pandas as pd

analysis_root = Path("/Users/michaelstaiger/Desktop/gitRepos/hablab_data_analysis/ifcb-metadata-export")
if str(analysis_root) not in sys.path:
    sys.path.insert(0, str(analysis_root))

from ifcb_pid_list_builders import get_ifcb_pids_by_daily_times
from ifcb_summary_streaming import summarize_bins_streaming

work_root = Path("/Users/michaelstaiger/Desktop/gitRepos/IFCBParticleSize/EmpyricalAnalysis/IFCBData/Test")
work_root.mkdir(parents=True, exist_ok=True)

pid_json_file = work_root / "mvco_test_list.json"
pid_csv_file = work_root / "mvco_test_list.csv"
summary_csv_file = work_root / "mvco_summary.csv"

print(f"Working folder: {work_root}")

Working folder: /Users/michaelstaiger/Desktop/gitRepos/IFCBParticleSize/EmpyricalAnalysis/IFCBData/Test


## 1. Build a PID list from the dashboard export

This uses the metadata export endpoint and keeps every PID in the selected time window. If you want representative times instead, the same module also exposes `get_ifcb_pids_by_daily_times`.

In [12]:
base_url = "https://habon-ifcb.whoi.edu"
dataset = "mvco"  # change this to your dataset slug
start = "2019-10-01T00:00:00Z"
end = "2019-10-30T23:59:59Z"
times = ["00:00", "06:00", "12:00", "18:00"]
same_day_only = True
include_skip = True

pids, picked_df = get_ifcb_pids_by_daily_times(
    base_url=base_url,
    dataset=dataset,
    start=start,
    end=end,
    times=times,
    same_day_only=same_day_only,
    include_skip=include_skip,
)

pid_json_file.write_text(json.dumps(pids, indent=2) + "\n", encoding="utf-8")
pid_csv_file.write_text(picked_df.to_csv(index=False), encoding="utf-8")

print(f"Found {len(pids)} PIDs")
print(f"Wrote JSON to {pid_json_file}")
pids[:10]
print(len(pids))

Found 100 PIDs
Wrote JSON to /Users/michaelstaiger/Desktop/gitRepos/IFCBParticleSize/EmpyricalAnalysis/IFCBData/Test/mvco_test_list.json
100


## 2. Build a summary dataset from the PID list

This uses the streaming summary utility to download each PID on demand, ingest the ADC and HDR files, and return one summary row per PID.

In [ ]:
## if you haev a preprepated pid list you can load it like this: 
#with open(pid_json_file, "r") as f:
 #   pids = json.load(f) 

In [14]:
dashboard_url = f"{base_url.rstrip('/')}/{dataset}"

summary_df = summarize_bins_streaming(
    dashboard_url=dashboard_url,
    pids=pids,
    drop_zero_roi=False,
    drop_false_trigger=True,
    max_roi_type=7,
    standardize_headers=True,
)

summary_df.to_csv(summary_csv_file, index=False)
print(f"Wrote summary CSV to {summary_csv_file}")
summary_df.head()

Wrote summary CSV to /Users/michaelstaiger/Desktop/gitRepos/IFCBParticleSize/EmpyricalAnalysis/IFCBData/Test/mvco_summary.csv


,roi0,roi1,roi2,roi3,roi4,roi5,roi6,roi7,vfinal,inhibittime_final,looktime,pid
0,0,4758,74,3,0,0,0,0,0.000269,0.000851,0.064453,D20191001T001056_IFCB010
1,106,4591,106,3,4,0,0,0,3.631015,326.181966,871.443709,D20191001T060059_IFCB010
2,104,4791,94,3,0,0,0,0,3.576966,339.328071,858.471792,D20191001T121254_IFCB010
3,103,4513,84,3,0,0,0,0,3.657835,319.739204,877.880512,D20191001T180257_IFCB010
4,128,5961,200,3,0,0,0,0,3.219209,425.241076,772.610052,D20191002T002136_IFCB010
